In [116]:
# Cell 0: Config
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy.stats import norm
import scipy.stats as stats
import numpy as np
from linearmodels.panel import PanelOLS

SPREADS_PATH = '../output/results/RESULTADOS A USAR/complete_results_weekly_25_dampening.csv'
DATE_COL     = 'date'
COUNTRY_COL  = 'country'
CDS_COL      = 'cds_spread'
YIELD_COL    = 'yield_market'
HORIZON = 5
RECOVERY = 0.4
HORIZONS     = [1, 2, 4, 8]
EXPORTERS    = ['Saudi Arabia', 'UAE (Abu Dhabi)', 'Colombia',
                'Mexico', 'Brazil', 'Egypt', 'Malaysia', 'Qatar']
CONTROLS     = ['Chile', 'China', 'Indonesia', 'Philippines', 'South Africa',
                'South Korea', 'Thailand', 'Turkey']

panel = pd.read_csv(SPREADS_PATH, parse_dates=[DATE_COL])

In [117]:
# Cell 2: Load controls and merge onto panel

# ── Load macro controls ───────────────────────────────────────
macro = pd.read_csv(
    '../data/processed/Macroeconomic_variables/macro_risk_variables.csv',
    sep=',', dayfirst=True, parse_dates=['Date'], index_col='Date'
)
vix = pd.read_csv(
    '../data/processed/Macroeconomic_variables/VIXCLS.csv',
    parse_dates=['Date'], index_col='Date'
)
ovx = pd.read_csv(
    '../data/processed/Macroeconomic_variables/OVXCLS.csv',
    parse_dates=['date'], index_col='date'
)
gpr = pd.read_csv(
    '../data/processed/Macroeconomic_variables/geopolitical_risk_index_daily.csv',
    sep=';', dayfirst=True, parse_dates=['date'], index_col='date',
    decimal=','
)
for col in gpr.columns:
    gpr[col] = pd.to_numeric(gpr[col], errors='coerce')

oil_prices  = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv',
                           parse_dates=['date'], index_col='date')
oil_futures = pd.read_csv('../data/processed/Oil/oil_futures.csv',
                           dayfirst=True, parse_dates=['date'], index_col='date')

# ── Build controls on daily index ────────────────────────────
macro['VIX']   = vix['VIXCLS']
macro['OVX']   = ovx['OVXCLS']
macro          = macro.join(gpr[['GPRD']], how='left')
macro['basis'] = oil_prices['Brent'] / oil_futures['Brent_12m']
controls_daily = macro[['VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD', 'basis']].copy()

# ── Load FX rates ─────────────────────────────────────────────
fx_wide = pd.read_csv('../data/processed/CCA_V2/exchange_rates.csv',
                       dayfirst=True, parse_dates=['date'])
fx = fx_wide.melt(id_vars='date', var_name=COUNTRY_COL, value_name='fx_rate')
fx = fx.dropna(subset=['fx_rate'])

fx[COUNTRY_COL] = fx[COUNTRY_COL].replace({'United Arab Emirates': 'UAE (Abu Dhabi)'})


# ── Anchor to panel dates ─────────────────────────────────────
cds_dates = pd.DatetimeIndex(panel[DATE_COL].unique())

# ── Reindex controls to panel dates ──────────────────────────
controls_weekly = controls_daily\
    .reindex(cds_dates, method='nearest',
             tolerance=pd.Timedelta('7 days'))\
    .ffill().bfill()\
    .reset_index()\
    .rename(columns={'Date': DATE_COL, 'index': DATE_COL})
controls_weekly[DATE_COL] = pd.to_datetime(controls_weekly[DATE_COL])

# ── Reindex FX per country to panel dates ────────────────────
fx_weekly = pd.merge_asof(
    panel[[DATE_COL, COUNTRY_COL]].sort_values(DATE_COL),
    fx.sort_values(DATE_COL),
    on=DATE_COL,
    by=COUNTRY_COL,
    direction='nearest',
    tolerance=pd.Timedelta('7 days')
)

fx_weekly

# ── Merge controls onto panel ─────────────────────────────────
panel = pd.merge_asof(
    panel.sort_values(DATE_COL),
    controls_weekly.sort_values(DATE_COL),
    on=DATE_COL,
    direction='nearest',
    tolerance=pd.Timedelta('7 days')
)

panel = panel.merge(
    fx_weekly[[DATE_COL, COUNTRY_COL, 'fx_rate']],
    on=[DATE_COL, COUNTRY_COL],
    how='left'
)

panel = panel.sort_values([COUNTRY_COL, DATE_COL]).reset_index(drop=True)

print("Panel shape:", panel.shape)
print("Columns:", panel.columns.tolist())
print("Nulls:\n", panel.isnull().sum()[panel.isnull().sum() > 0])


Panel shape: (8352, 33)
Columns: ['date', 'country', 'cds_spread', 'risk_free_rate', 'B_f', 'LCL_usd', 'sigma_lcl', 'implied_V_M0', 'implied_sigma_V_M0', 'cca_converged_M0', 'implied_V_M1', 'implied_sigma_V_M1', 'cca_converged_M1', 'convenience_yield', 'implied_V_M2', 'implied_sigma_V_M2', 'cca_converged_M2', 'lambda_annual', 'group', 'exporter', 'DD_M0', 'DD_M1', 'DD_M2', 'PD_M0', 'PD_M1', 'PD_M2', 'VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD', 'basis', 'fx_rate']
Nulls:
 Series([], dtype: int64)


/var/folders/wy/gjw_3_n51t748hfngpz4zf0w0000gn/T/ipykernel_70828/879418672.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  oil_prices  = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv',


# 0: Correlations

In [118]:
# Correlation of levels: CDS vs DD across models
from scipy.stats import pearsonr, spearmanr

print(f"{'Country':<20} | {'M0 Pearson':>10} {'p':>8} {'Spearman':>10} {'p':>8} | {'M1 Pearson':>10} {'p':>8} {'Spearman':>10} {'p':>8} | {'M2 Pearson':>10} {'p':>8} {'Spearman':>10} {'p':>8}")
print("-" * 130)

corrs = {f'{m}_{g}_{t}': [] for m in ['M0','M1','M2'] for g in ['exp','ctl'] for t in ['pear','spear']}

for c in EXPORTERS + CONTROLS:
    grp = 'exp' if c in EXPORTERS else 'ctl'
    sub = panel[panel[COUNTRY_COL] == c].copy()
    
    line = f"{c:<20}"
    for model, dd_col in [('M0', 'DD_M0'), ('M1', 'DD_M1'), ('M2', 'DD_M2')]:
        tmp = sub[[CDS_COL, dd_col]].dropna()
        tmp = tmp[tmp[CDS_COL] > 0]
        
        if len(tmp) > 10:
            pr, pp = pearsonr(tmp[CDS_COL], tmp[dd_col])
            sr, sp = spearmanr(tmp[CDS_COL], tmp[dd_col])
        else:
            pr, pp, sr, sp = np.nan, np.nan, np.nan, np.nan
        
        corrs[f'{model}_{grp}_pear'].append(pr)
        corrs[f'{model}_{grp}_spear'].append(sr)
        
        p_star = '***' if pp < 0.01 else '**' if pp < 0.05 else '*' if pp < 0.1 else ''
        s_star = '***' if sp < 0.01 else '**' if sp < 0.05 else '*' if sp < 0.1 else ''
        line += f" | {pr:>9.3f}{p_star:<3} {pp:>8.4f} {sr:>9.3f}{s_star:<3} {sp:>8.4f}"
    
    print(line)

print("-" * 130)
for g, label in [('exp', 'Exporters'), ('ctl', 'Controls')]:
    line = f"Mean {label:<15}"
    for m in ['M0', 'M1', 'M2']:
        mp = np.nanmean(corrs[f'{m}_{g}_pear'])
        ms = np.nanmean(corrs[f'{m}_{g}_spear'])
        line += f" | {mp:>10.3f} {'':>8} {ms:>10.3f} {'':>8}"
    print(line)

Country              | M0 Pearson        p   Spearman        p | M1 Pearson        p   Spearman        p | M2 Pearson        p   Spearman        p
----------------------------------------------------------------------------------------------------------------------------------
Saudi Arabia         |    -0.605***   0.0000    -0.738***   0.0000 |    -0.658***   0.0000    -0.805***   0.0000 |    -0.703***   0.0000    -0.767***   0.0000
UAE (Abu Dhabi)      |     0.155***   0.0004     0.123***   0.0049 |    -0.486***   0.0000    -0.407***   0.0000 |    -0.452***   0.0000    -0.449***   0.0000
Colombia             |    -0.546***   0.0000    -0.558***   0.0000 |    -0.469***   0.0000    -0.501***   0.0000 |    -0.586***   0.0000    -0.608***   0.0000
Mexico               |    -0.379***   0.0000    -0.379***   0.0000 |    -0.478***   0.0000    -0.403***   0.0000 |    -0.557***   0.0000    -0.504***   0.0000
Brazil               |    -0.524***   0.0000    -0.532***   0.0000 |    -0.586***   0.

# 1. Inseparability Test: OVX and Futures Basis vs Global Risk Factors

In [119]:
# Use unique dates from panel — one row per date for time series regressions
ts = panel.drop_duplicates(subset=[DATE_COL]).set_index(DATE_COL)\
          [['VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD', 'basis']].sort_index()

# ── OVX ~ Global factors (levels) ────────────────────────────
idx   = ts[['OVX', 'VIX', 'DXY', 'UST10Y', 'GPRD']].dropna().index
y_ovx = ts.loc[idx, 'OVX']
X_ovx = sm.add_constant(ts.loc[idx, ['VIX', 'DXY', 'UST10Y', 'GPRD']])
res_ovx = sm.OLS(y_ovx, X_ovx).fit(cov_type='HAC', cov_kwds={'maxlags': 1})

print("OVX ~ Global Factors (levels)")
print(f"R²: {res_ovx.rsquared:.4f}  |  Adj. R²: {res_ovx.rsquared_adj:.4f}")
print(f"\n{'Variable':<12} {'Beta':>10} {'p-value':>10}")
print("-" * 35)
for var in X_ovx.columns:
    print(f"{var:<12} {res_ovx.params[var]:>10.4f} {res_ovx.pvalues[var]:>10.4f}")

OVX ~ Global Factors (levels)
R²: 0.5903  |  Adj. R²: 0.5872

Variable           Beta    p-value
-----------------------------------
const         -101.0461     0.0000
VIX              1.4614     0.0000
DXY              1.3520     0.0000
UST10Y          -7.6438     0.0000
GPRD             0.0104     0.4554


In [120]:
# Convenience yield ~ Global Factors
cy = panel.drop_duplicates(subset=[DATE_COL]).set_index(DATE_COL)[['convenience_yield', 'VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD']].sort_index().dropna()

y = cy['convenience_yield']
X = sm.add_constant(cy[['VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD']])
res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 1})

print("Convenience Yield ~ Global Factors (levels)")
print(f"R²: {res.rsquared:.4f}  |  Adj. R²: {res.rsquared_adj:.4f}")
print(f"\n{'Variable':<15} {'Beta':>10} {'p-value':>10}")
print("-" * 38)
for var in X.columns:
    print(f"{var:<15} {res.params[var]:>10.4f} {res.pvalues[var]:>10.4f}")

Convenience Yield ~ Global Factors (levels)
R²: 0.5105  |  Adj. R²: 0.5057

Variable              Beta    p-value
--------------------------------------
const              -0.2401     0.0180
VIX                 0.0056     0.0000
OVX                -0.0041     0.0000
DXY                 0.0027     0.0302
UST10Y              0.0208     0.0004
GPRD                0.0002     0.0449


# 2. Regressions

In [121]:
# ============================================================
# Regression 1: ln(CDS) = α + β·ln(V) + ε  — Asset Value (M0 vs M1)
# ============================================================

print("Regression 1: ln(CDS) = α + β·ln(V) + ε\n")
print(f"{'Country':<20} | {'M0 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M1 β':>8} {'p':>8} {'R²':>7} {'N':>5}")
print("-" * 90)

r2_v = {'M0_exp': [], 'M1_exp': [], 'M0_ctl': [], 'M1_ctl': []}

for c in EXPORTERS + CONTROLS:
    grp = 'exp' if c in EXPORTERS else 'ctl'
    line = f"{c:<20}"

    for model, v_col in [('M0', 'implied_V_M0'), ('M1', 'implied_V_M1')]:
        df = panel[panel[COUNTRY_COL] == c][[CDS_COL, v_col]].dropna()
        df = df[(df[CDS_COL] > 0) & (df[v_col] > 0)]
        y = np.log(df[CDS_COL])
        X = sm.add_constant(np.log(df[v_col]))
        res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 8})
        b = res.params.iloc[1]
        p = res.pvalues.iloc[1]
        sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
        line += f" | {b:>7.3f}{sig:<3} {p:>7.4f} {res.rsquared:>7.3f} {len(df):>5}"
        r2_v[f'{model}_{grp}'].append(res.rsquared)

    print(line)

print("-" * 90)
print(f"{'Mean R² Exporters':<20} | {'':>8} {'':>8} {np.mean(r2_v['M0_exp']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_v['M1_exp']):>7.3f}")
print(f"{'Mean R² Controls':<20} | {'':>8} {'':>8} {np.mean(r2_v['M0_ctl']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_v['M1_ctl']):>7.3f}")

Regression 1: ln(CDS) = α + β·ln(V) + ε

Country              |     M0 β        p      R²     N |     M1 β        p      R²     N
------------------------------------------------------------------------------------------
Saudi Arabia         |  -0.575***  0.0000   0.484   522 |  -0.509***  0.0000   0.553   522
UAE (Abu Dhabi)      |  -0.613***  0.0000   0.389   522 |  -0.505***  0.0000   0.460   522
Colombia             |   0.384*    0.0597   0.043   522 |   0.132     0.3573   0.012   522
Mexico               |  -0.861***  0.0000   0.188   522 |  -0.583***  0.0000   0.293   522
Brazil               |  -3.559***  0.0000   0.500   522 |  -1.154***  0.0000   0.396   522
Egypt                |   1.167***  0.0000   0.279   522 |   0.893***  0.0000   0.280   522
Malaysia             |  -2.616***  0.0000   0.519   522 |  -1.513***  0.0000   0.531   522
Qatar                |  -0.814***  0.0000   0.573   522 |  -0.664***  0.0000   0.591   522
Chile                |  -0.248     0.2492   0.018  

In [122]:
# ============================================================
# Regression 2: ln(CDS) = α + β·ln(σ) + ε  — Volatility (M0 vs M2)
# ============================================================
 
print("\n\nRegression 2: ln(CDS) = α + β·ln(σ) + ε\n")
print(f"{'Country':<20} | {'M0 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M2 β':>8} {'p':>8} {'R²':>7} {'N':>5}")
print("-" * 90)
 
r2_vol = {'M0_exp': [], 'M2_exp': [], 'M0_ctl': [], 'M2_ctl': []}
 
for c in EXPORTERS + CONTROLS:
    grp = 'exp' if c in EXPORTERS else 'ctl'
    line = f"{c:<20}"
 
    for model, sig_col in [('M0', 'implied_sigma_V_M0'), ('M2', 'implied_sigma_V_M2')]:
        df = panel[panel[COUNTRY_COL] == c][[CDS_COL, sig_col]].dropna()
        df = df[(df[CDS_COL] > 0) & (df[sig_col] > 0)]
        y = np.log(df[CDS_COL])
        X = sm.add_constant(np.log(df[sig_col]))
        res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 8})
        b = res.params.iloc[1]
        p = res.pvalues.iloc[1]
        sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
        line += f" | {b:>7.3f}{sig:<3} {p:>7.4f} {res.rsquared:>7.3f} {len(df):>5}"
        r2_vol[f'{model}_{grp}'].append(res.rsquared)
 
    print(line)
 
print("-" * 90)
print(f"{'Mean R² Exporters':<20} | {'':>8} {'':>8} {np.mean(r2_vol['M0_exp']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_vol['M2_exp']):>7.3f}")
print(f"{'Mean R² Controls':<20} | {'':>8} {'':>8} {np.mean(r2_vol['M0_ctl']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_vol['M2_ctl']):>7.3f}")



Regression 2: ln(CDS) = α + β·ln(σ) + ε

Country              |     M0 β        p      R²     N |     M2 β        p      R²     N
------------------------------------------------------------------------------------------
Saudi Arabia         |   0.416***  0.0000   0.577   522 |   0.784***  0.0000   0.675   522
UAE (Abu Dhabi)      |  -0.100**   0.0483   0.055   522 |   0.088     0.6175   0.003   522
Colombia             |   0.622***  0.0020   0.193   522 |   0.875***  0.0006   0.227   522
Mexico               |   0.047     0.7228   0.002   522 |   0.567**   0.0201   0.106   522
Brazil               |   0.694***  0.0005   0.168   522 |   1.198***  0.0000   0.279   522
Egypt                |   0.153**   0.0446   0.040   522 |   0.169*    0.0927   0.028   522
Malaysia             |   0.397*    0.0702   0.042   522 |   1.586***  0.0005   0.154   522
Qatar                |   0.403***  0.0000   0.504   522 |   0.865***  0.0000   0.615   522
Chile                |   0.131     0.3148   0.013

In [123]:
print("Regression: ln(CDS) = α + β·DD + ε\n")
print(f"{'Country':<20} | {'M0 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M1 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M2 β':>8} {'p':>8} {'R²':>7} {'N':>5}")
print("-" * 120)

r2 = {'M0_exp': [], 'M1_exp': [], 'M2_exp': [], 'M0_ctl': [], 'M1_ctl': [], 'M2_ctl': []}
betas = {'M0_exp': [], 'M1_exp': [], 'M2_exp': [], 'M0_ctl': [], 'M1_ctl': [], 'M2_ctl': []}

for c in EXPORTERS + CONTROLS:
    grp = 'exp' if c in EXPORTERS else 'ctl'
    line = f"{c:<20}"
    
    for model, dd_col in [('M0', 'DD_M0'), ('M1', 'DD_M1'), ('M2', 'DD_M2')]:
        df = panel[panel[COUNTRY_COL] == c][[CDS_COL, dd_col]].dropna()
        df = df[df[CDS_COL] > 0]
        y = np.log(df[CDS_COL].values)
        X = sm.add_constant(df[dd_col].values)
        res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 8})
        
        b = res.params[1]
        p = res.pvalues[1]
        sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
        
        line += f" | {b:>7.3f}{sig:<3} {p:>7.4f} {res.rsquared:>7.3f} {len(df):>5}"
        
        r2[f'{model}_{grp}'].append(res.rsquared)
        betas[f'{model}_{grp}'].append(b)
    
    print(line)

print("-" * 120)

print(
    f"{'Mean β Exporters':<20} | "
    f"{np.mean(betas['M0_exp']):>7.3f} {'':>8} {'':>7} {'':>5} | "
    f"{np.mean(betas['M1_exp']):>7.3f} {'':>8} {'':>7} {'':>5} | "
    f"{np.mean(betas['M2_exp']):>7.3f}"
)
print(
    f"{'Mean β Controls':<20} | "
    f"{np.mean(betas['M0_ctl']):>7.3f} {'':>8} {'':>7} {'':>5} | "
    f"{np.mean(betas['M1_ctl']):>7.3f} {'':>8} {'':>7} {'':>5} | "
    f"{np.mean(betas['M2_ctl']):>7.3f}"
)
print(
    f"{'Mean R² Exporters':<20} | "
    f"{'':>8} {'':>8} {np.mean(r2['M0_exp']):>7.3f} {'':>5} | "
    f"{'':>8} {'':>8} {np.mean(r2['M1_exp']):>7.3f} {'':>5} | "
    f"{'':>8} {'':>8} {np.mean(r2['M2_exp']):>7.3f}"
)
print(
    f"{'Mean R² Controls':<20} | "
    f"{'':>8} {'':>8} {np.mean(r2['M0_ctl']):>7.3f} {'':>5} | "
    f"{'':>8} {'':>8} {np.mean(r2['M1_ctl']):>7.3f} {'':>5} | "
    f"{'':>8} {'':>8} {np.mean(r2['M2_ctl']):>7.3f}"
)

Regression: ln(CDS) = α + β·DD + ε

Country              |     M0 β        p      R²     N |     M1 β        p      R²     N |     M2 β        p      R²     N
------------------------------------------------------------------------------------------------------------------------
Saudi Arabia         |  -0.110***  0.0000   0.448   522 |  -0.105***  0.0000   0.524   522 |  -0.457***  0.0000   0.547   522
UAE (Abu Dhabi)      |   0.009     0.3787   0.014   522 |  -0.066***  0.0002   0.222   522 |  -0.450***  0.0001   0.217   522
Colombia             |  -0.238***  0.0000   0.309   522 |  -0.159***  0.0000   0.248   522 |  -0.445***  0.0000   0.336   522
Mexico               |  -0.111***  0.0090   0.139   522 |  -0.065***  0.0016   0.207   522 |  -0.286***  0.0000   0.292   522
Brazil               |  -0.202***  0.0000   0.297   522 |  -0.145***  0.0000   0.352   522 |  -0.393***  0.0000   0.401   522
Egypt                |  -0.074***  0.0000   0.214   522 |  -0.062***  0.0000   0.157   522

In [124]:
print("Regression: ln(CDS) = α + β·DD + ln(VIX) + ln(UST10Y) + ln(DXY) + ln(GPRD) + ln(FX) + ε\n")
print(f"{'Country':<20} | {'M0 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M1 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M2 β':>8} {'p':>8} {'R²':>7} {'N':>5}")
print("-" * 120)
r2_c = {'M0_exp': [], 'M1_exp': [], 'M2_exp': [], 'M0_ctl': [], 'M1_ctl': [], 'M2_ctl': []}

for c in EXPORTERS + CONTROLS:
    grp = 'exp' if c in EXPORTERS else 'ctl'
    line = f"{c:<20}"
    
    for model, dd_col in [('M0', 'DD_M0'), ('M1', 'DD_M1'), ('M2', 'DD_M2')]:
        df = panel[panel[COUNTRY_COL] == c][[CDS_COL, dd_col, 'VIX', 'UST10Y', 'DXY', 'GPRD', 'fx_rate']].dropna()
        df = df[(df[CDS_COL] > 0) & (df['VIX'] > 0) & (df['UST10Y'] > 0) & (df['DXY'] > 0) & (df['GPRD'] > 0) & (df['fx_rate'] > 0)].reset_index(drop=True)
        
        if len(df) < 30:
            line += f" | {'--':>7}    {'--':>7} {'--':>7} {len(df):>5}"
            continue
        
        y = np.log(df[CDS_COL].values)
        X = np.column_stack([
            np.ones(len(df)),
            df[dd_col].values,
            np.log(df['VIX'].values),
            np.log(df['UST10Y'].values),
            np.log(df['DXY'].values),
            np.log(df['GPRD'].values),
            np.log(df['fx_rate'].values)
        ])
        
        if not np.all(np.isfinite(X)) or not np.all(np.isfinite(y)):
            line += f" | {'--':>7}    {'--':>7} {'--':>7} {len(df):>5}"
            continue
        
        res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 8})
        b = res.params[1]
        p = res.pvalues[1]
        sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
        line += f" | {b:>7.3f}{sig:<3} {p:>7.4f} {res.rsquared:>7.3f} {len(df):>5}"
        r2_c[f'{model}_{grp}'].append(res.rsquared)
    
    print(line)

print("-" * 120)
print(f"{'Mean R² Exporters':<20} | {'':>8} {'':>8} {np.mean(r2_c['M0_exp']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_c['M1_exp']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_c['M2_exp']):>7.3f}")
print(f"{'Mean R² Controls':<20} | {'':>8} {'':>8} {np.mean(r2_c['M0_ctl']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_c['M1_ctl']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_c['M2_ctl']):>7.3f}")

Regression: ln(CDS) = α + β·DD + ln(VIX) + ln(UST10Y) + ln(DXY) + ln(GPRD) + ln(FX) + ε

Country              |     M0 β        p      R²     N |     M1 β        p      R²     N |     M2 β        p      R²     N
------------------------------------------------------------------------------------------------------------------------
Saudi Arabia         |  -0.118***  0.0000   0.496   522 |  -0.105***  0.0000   0.545   522 |  -0.439***  0.0000   0.551   522
UAE (Abu Dhabi)      |   0.003     0.8160   0.102   522 |  -0.091***  0.0002   0.245   522 |  -0.430***  0.0013   0.230   522
Colombia             |  -0.201***  0.0000   0.609   522 |  -0.174***  0.0000   0.670   522 |  -0.471***  0.0000   0.639   522
Mexico               |  -0.120***  0.0022   0.332   522 |  -0.088***  0.0000   0.437   522 |  -0.531***  0.0000   0.611   522
Brazil               |  -0.249***  0.0000   0.492   522 |  -0.215***  0.0000   0.553   522 |  -0.525***  0.0000   0.575   522
Egypt                |  -0.024*    0.

## Pooled Regression

In [128]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf


MODEL_MAP = {
    'M0': 'DD_M0',
    'M1': 'DD_M1',
    'M2': 'DD_M2'
}

print("Pooled FE regression: ln(CDS) = α_i + λ_t + β·DD + γ·(DD×Exporter) + ε\n")
print(f"{'Model':<6} | {'β control':>10} {'p':>8} | {'γ exporter diff':>16} {'p':>8} | {'β exporter':>11} {'p':>8} | {'R²':>7} {'N':>6}")
print("-" * 95)

for model_name, dd_col in MODEL_MAP.items():
    df = panel[[COUNTRY_COL, DATE_COL, CDS_COL, dd_col]].dropna().copy()
    df = df[df[CDS_COL] > 0].copy()

    df['ln_cds'] = np.log(df[CDS_COL])
    df['exporter'] = df[COUNTRY_COL].isin(EXPORTERS).astype(int)
    df['dd_exporter'] = df[dd_col] * df['exporter']

    formula = f"ln_cds ~ {dd_col} + dd_exporter + C({COUNTRY_COL}) + C({DATE_COL})"
    res = smf.ols(formula, data=df).fit(
        cov_type='HAC',
        cov_kwds={'maxlags': 12}
    )

    beta = res.params[dd_col]
    p_beta = res.pvalues[dd_col]

    gamma = res.params['dd_exporter']
    p_gamma = res.pvalues['dd_exporter']

    beta_exporter = beta + gamma
    p_exporter = float(res.t_test(f"{dd_col} + dd_exporter = 0").pvalue)

    print(
        f"{model_name:<6} | "
        f"{beta:>10.4f} {p_beta:>8.4f} | "
        f"{gamma:>16.4f} {p_gamma:>8.4f} | "
        f"{beta_exporter:>11.4f} {p_exporter:>8.4f} | "
        f"{res.rsquared:>7.3f} {int(res.nobs):>6}"
    )

Pooled FE regression: ln(CDS) = α_i + λ_t + β·DD + γ·(DD×Exporter) + ε

Model  |  β control        p |  γ exporter diff        p |  β exporter        p |      R²      N
-----------------------------------------------------------------------------------------------
M0     |    -0.0153   0.1297 |          -0.0391   0.0049 |     -0.0544   0.0000 |   0.899   8352
M1     |    -0.0423   0.0000 |          -0.0364   0.0000 |     -0.0786   0.0000 |   0.906   8352
M2     |    -0.1134   0.0003 |          -0.0705   0.0775 |     -0.1839   0.0000 |   0.905   8352


In [130]:
from linearmodels.panel import PanelOLS

for model_name, dd_col in MODEL_MAP.items():
    df = panel[[COUNTRY_COL, DATE_COL, CDS_COL, dd_col]].dropna().copy()
    df = df[df[CDS_COL] > 0].copy()
    df['ln_cds'] = np.log(df[CDS_COL])
    df['exporter'] = df[COUNTRY_COL].isin(EXPORTERS).astype(int)
    df['dd_exporter'] = df[dd_col] * df['exporter']
    df = df.set_index([COUNTRY_COL, DATE_COL])

    mod = PanelOLS(df['ln_cds'], df[[dd_col, 'dd_exporter']], entity_effects=True, time_effects=True)
    res = mod.fit(cov_type='kernel', kernel='bartlett', bandwidth=12)
    print(res.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                 ln_cds   R-squared:                        0.0967
Estimator:                   PanelOLS   R-squared (Between):             -0.0712
No. Observations:                8352   R-squared (Within):               0.0609
Date:                Sat, Apr 11 2026   R-squared (Overall):             -0.0704
Time:                        00:46:35   Log-likelihood                   -257.37
Cov. Estimator:        Driscoll-Kraay                                           
                                        F-statistic:                      418.11
Entities:                          16   P-value                           0.0000
Avg Obs:                       522.00   Distribution:                  F(2,7813)
Min Obs:                       522.00                                           
Max Obs:                       522.00   F-statistic (robust):             11.751
                            